In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import joblib
import folium
import config as cfg

db  = joblib.load('../models/dbscan.pkl')
iso = joblib.load('../models/isoforest.pkl')

print("Models loaded successfully")
print("DBSCAN  :", db)
print("IsoForest:", iso)

C:\Users\Visha\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Visha\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Models loaded successfully
DBSCAN  : DBSCAN(eps=0.003, metric='haversine')
IsoForest: IsolationForest(contamination=0.05, random_state=42)


In [2]:
def score_location(lat, lng, hour):
    # build feature row
    lat_rad  = np.radians(lat)
    lng_rad  = np.radians(lng)
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)

    point = np.array([[lat_rad, lng_rad, hour_sin, hour_cos]])

    # DBSCAN — check if inside any known cluster
    coords      = np.array([[lat_rad, lng_rad]])
    core_coords = db.components_
    from sklearn.metrics.pairwise import haversine_distances
    if len(core_coords) > 0:
        dists       = haversine_distances(coords, core_coords)[0]
        min_dist_km = dists.min() * 6371
        in_cluster  = min_dist_km < (cfg.DBSCAN_EPS * 6371)
    else:
        in_cluster  = False
        min_dist_km = 999

    # IsoForest score
    iso_score = iso.decision_function(point)[0]
    iso_pred  = iso.predict(point)[0]

    # combined risk
    is_anomaly = (not in_cluster) or (iso_pred == -1)
    risk = round((1 - (iso_score + 0.5)) * 50 + (min_dist_km * 10), 2)
    risk = min(max(risk, 0), 100)

    return {
        "lat"         : lat,
        "lng"         : lng,
        "hour"        : hour,
        "dist_km"     : round(min_dist_km, 3),
        "iso_score"   : round(iso_score, 4),
        "in_cluster"  : in_cluster,
        "is_anomaly"  : is_anomaly,
        "risk_score"  : risk,
        "status"      : "ANOMALY" if is_anomaly else "SAFE"
    }

print("Scoring function ready")

Scoring function ready


In [3]:
test_points = [
    {"name": "Railway Station (daytime)",  "lat": 18.1115, "lng": 83.3963, "hour": 10},
    {"name": "Bus Stand (evening)",        "lat": 18.1099, "lng": 83.3994, "hour": 18},
    {"name": "Fort Park (midnight)",       "lat": 18.1116, "lng": 83.4096, "hour": 0 },
    {"name": "Unknown area (night)",       "lat": 18.08,   "lng": 83.45,   "hour": 2 },
    {"name": "Far outside city",           "lat": 18.05,   "lng": 83.35,   "hour": 23},
]

results = []
for p in test_points:
    r = score_location(p['lat'], p['lng'], p['hour'])
    r['name'] = p['name']
    results.append(r)

df_results = pd.DataFrame(results)[['name','dist_km','risk_score','in_cluster','is_anomaly','status']]
df_results

,name,dist_km,risk_score,in_cluster,is_anomaly,status
0,Railway Station (daytime),0.005,18.79,True,False,SAFE
1,Bus Stand (evening),0.043,20.06,True,False,SAFE
2,Fort Park (midnight),0.052,25.51,True,False,SAFE
3,Unknown area (night),4.394,72.94,True,True,ANOMALY
4,Far outside city,7.257,100.00,True,True,ANOMALY


In [4]:
from IPython.display import display, HTML

display(HTML("""
<div style="padding:12px;background:#1e1e1e;border-radius:8px">
<button onclick="
  navigator.geolocation.getCurrentPosition(function(p){
    document.getElementById('coords').innerText =
      'lat=' + p.coords.latitude.toFixed(6) +
      '  lng=' + p.coords.longitude.toFixed(6);
  }, function(e){ document.getElementById('coords').innerText='Error: allow location access'; })
" style="padding:8px 16px;background:#4CAF50;color:white;border:none;border-radius:4px;cursor:pointer;font-size:14px">
  Get My Current Location
</button>
<p id="coords" style="color:#90EE90;margin-top:8px;font-family:monospace">
  Click button above...
</p>
</div>
"""))

In [5]:
# After clicking the button above, copy the lat and lng shown
# and paste them below

MY_CURRENT_LAT  = 18.101439   # replace with your actual lat
MY_CURRENT_LNG  = 83.420569  # replace with your actual lng
MY_CURRENT_HOUR = 15       # replace with current hour (24h format)

result = score_location(MY_CURRENT_LAT, MY_CURRENT_LNG, MY_CURRENT_HOUR)

print("=" * 40)
print(f"  Location  : {result['lat']}, {result['lng']}")
print(f"  Hour      : {result['hour']}:00")
print(f"  Distance  : {result['dist_km']} km from nearest safe zone")
print(f"  Risk Score: {result['risk_score']} / 100")
print(f"  Status    : {result['status']}")
print("=" * 40)

if result['is_anomaly']:
    print("  WARNING: You are outside known safe zones!")
else:
    print("  You are in a safe known area.")

  Location  : 18.101439, 83.420569
  Hour      : 15:00
  Distance  : 0.598 km from nearest safe zone
  Risk Score: 30.7 / 100
  Status    : SAFE
  You are in a safe known area.


In [6]:
m2 = folium.Map(location=[MY_CURRENT_LAT, MY_CURRENT_LNG], zoom_start=14)

# safe zone circles around each locality
for name, coords in cfg.LOCALITIES.items():
    folium.Circle(
        location=list(coords),
        radius=300,
        color='green',
        fill=True,
        fill_opacity=0.2,
        popup=name.replace('_',' ').title()
    ).add_to(m2)
    folium.Marker(
        location=list(coords),
        tooltip=name.replace('_',' ').title(),
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(m2)

# your current position
marker_color = 'red' if result['is_anomaly'] else 'green'
folium.Marker(
    location=[MY_CURRENT_LAT, MY_CURRENT_LNG],
    popup=f"YOU ARE HERE\nRisk: {result['risk_score']}/100\nStatus: {result['status']}",
    tooltip="Your location",
    icon=folium.Icon(color=marker_color, icon='user')
).add_to(m2)

m2.save('../maps/live_location_map.html')
print("Saved to maps/live_location_map.html")
m2

Saved to maps/live_location_map.html
